# Phase 2: AdaFoB Hypothesis Validation Pilot

This notebook validates two core hypotheses of the AdaFoB project using **frozen SAM ViT-H** inference (no training):

- **Claim A**: A shape-adaptive Np beats every fixed Np on irregular structures.
- **Claim B**: Skeleton-derived background prompt placement beats the ring/dilation-band prior on HD95 for irregular structures.

**Hardware**: Kaggle GPU T4 x2 (uses only `cuda:0`).

---

## Dataset Setup Instructions

### A) BraTS2020 (Required - Irregular Structure Benchmark)
1. In the Kaggle Notebook editor, click **Add Input** (right sidebar).
2. Search for **"BraTS2020"** and add the dataset (e.g., `awsaf49/brats20-dataset-training-validation`).
3. It will mount read-only at `/kaggle/input/<dataset-slug>/`.
4. **Label convention**: Tumor core = union of labels {1, 4} (necrotic + enhancing tumor). Label 2 = edema (excluded). Label 3 is absent in BraTS2020.

### B) Abd-CT / BTCV (Required - Compact Control)
**Option 1 (Recommended)**: Search Kaggle Datasets for **"BTCV abdomen CT"** or **"Synapse multi-organ"** and add as input. Document the exact slug in the path-resolution cell below.

**Option 2 (Synapse download)**: If no Kaggle mirror is available:
1. Register at https://www.synapse.org and join syn3193805.
2. Add your Synapse Personal Access Token as Kaggle Secret `SYNAPSE_TOKEN`.
3. Uncomment the Synapse download cell below.

### C) DRIVE (Optional - Cross-Domain Appendix)
1. **Add Input** -> search **"DRIVE retinal vessel segmentation"** -> add.
2. If skipped, leave `DRIVE_ROOT = None` -- the scripts handle this gracefully.

### D) GitHub Token (if repo is private)
If the repo is private, add a GitHub Personal Access Token as Kaggle Secret `GITHUB_TOKEN`. The clone command below will use it automatically.

---
## 1. Environment Setup

In [ ]:
import os

# Clone the repo
repo_dir = '/kaggle/working/AdaFoB'

if not os.path.isdir(repo_dir):
    # If repo is private, use GITHUB_TOKEN secret:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        token = secrets.get_secret('GITHUB_TOKEN')
        !git clone https://{token}@github.com/PhoenixEvo/AdaFoB.git {repo_dir}
    except Exception:
        # Public repo fallback
        !git clone https://github.com/PhoenixEvo/AdaFoB.git {repo_dir}
else:
    # Pull latest
    !cd {repo_dir} && git pull

%cd {repo_dir}
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install additional packages not preinstalled on Kaggle
!pip install -q segment-anything medpy skan pyyaml

In [ ]:
# Download SAM ViT-H checkpoint
!mkdir -p /kaggle/working/checkpoints
!wget -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth \
    -P /kaggle/working/checkpoints/
!ls -lh /kaggle/working/checkpoints/

---
## 2. Path Resolution

Set dataset paths to match your Kaggle input mounts. Edit the slugs below to match what you added.

In [ ]:
import os
import glob

# Auto-detect dataset paths dynamically in /kaggle/input
BRATS_ROOT = None
ABDCT_ROOT = None
DRIVE_ROOT = None

# Search for BraTS
brats_candidates = glob.glob('/kaggle/input/**/*BraTS20_Training_*', recursive=True)
if brats_candidates:
    BRATS_ROOT = os.path.dirname(brats_candidates[0])
    print(f'BraTS root auto-detected: {BRATS_ROOT}')
else:
    print(f'[WARN] BraTS not found in /kaggle/input')

# Search for Abd-CT
abdct_candidates = glob.glob('/kaggle/input/**/*averaged-training-images*', recursive=True)
if abdct_candidates:
    ABDCT_ROOT = os.path.dirname(abdct_candidates[0])
    print(f'AbdCT root auto-detected: {ABDCT_ROOT}')
else:
    print(f'[WARN] Abd-CT not found in /kaggle/input')

# Search for DRIVE
drive_candidates = glob.glob('/kaggle/input/**/*DRIVE*', recursive=True)
if drive_candidates:
    DRIVE_ROOT = drive_candidates[0]
    print(f'DRIVE root auto-detected: {DRIVE_ROOT}')
else:
    print(f'[SKIP] DRIVE not found in /kaggle/input')

# ---- SAM checkpoint ----
SAM_CHECKPOINT = '/kaggle/working/checkpoints/sam_vit_h_4b8939.pth'

# Set environment variables
os.environ['BRATS_ROOT'] = BRATS_ROOT or ''
os.environ['ABDCT_ROOT'] = ABDCT_ROOT or ''
os.environ['DRIVE_ROOT'] = DRIVE_ROOT or ''
os.environ['SAM_CHECKPOINT'] = SAM_CHECKPOINT

print('\\nEnvironment variables set.')
\n

In [ ]:
# Optional: Synapse download (uncomment if BTCV mirror is not available)
# !pip install -q synapseclient
# import synapseclient
# from kaggle_secrets import UserSecretsClient
# secrets = UserSecretsClient()
# syn = synapseclient.Synapse()
# syn.login(authToken=secrets.get_secret('SYNAPSE_TOKEN'))
# 
# # Download specific SABS files - adjust entity IDs as needed
# import os
# os.makedirs('/kaggle/working/data/abdct_pilot', exist_ok=True)
# # syn.get('synXXXXXXX', downloadLocation='/kaggle/working/data/abdct_pilot')
# print('Synapse download complete.')

---
## 3. Sanity Check: Shape Features

Quick test on synthetic masks to verify the shape feature computation works.

In [ ]:
import time
t0 = time.time()

# Run the built-in sanity check
!python experiments/pilot/shape_features.py

elapsed = time.time() - t0
print(f'\nShape features sanity check elapsed: {elapsed:.1f}s')

---
## 4. Collect Shape Features

Runs `collect_shape_features.py` to extract features from all pilot cases and save to CSV.

In [ ]:
import time
import torch

t0 = time.time()
if torch.cuda.is_available():
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    start_event.record()

!python experiments/pilot/collect_shape_features.py

if torch.cuda.is_available():
    end_event.record()
    torch.cuda.synchronize()
    gpu_ms = start_event.elapsed_time(end_event)
    print(f'GPU time: {gpu_ms/1000:.1f}s')

elapsed = time.time() - t0
print(f'Wall time: {elapsed:.1f}s')

In [ ]:
# Inspect the shape features CSV
import pandas as pd
df = pd.read_csv('results/pilot/pilot_shape_features.csv')
print(f'Total cases: {len(df)}')
print(f'\nDataset distribution:')
print(df['dataset'].value_counts())
print(f'\nNp_heuristic stats:')
print(df.groupby('dataset')['Np_heuristic'].describe())
df

---
## 5. Run Pilot Comparison

Runs `run_pilot.py` -- the main experiment. This loads SAM ViT-H and evaluates all 6 arms.

In [ ]:
import time

t0 = time.time()

!python experiments/pilot/run_pilot.py

elapsed = time.time() - t0
print(f'\nTotal wall time for pilot: {elapsed:.1f}s ({elapsed/3600:.3f} hours)')

In [ ]:
# Show GPU timing
with open('logs/pilot_gpu_time.txt', 'r') as f:
    print(f.read())

---
## 6. Results

In [ ]:
import pandas as pd

# Per-case metrics
metrics = pd.read_csv('results/pilot/pilot_metrics.csv')
print(f'Total rows: {len(metrics)}')
print(f'Cases: {metrics["case_id"].nunique()}')
print(f'Arms: {metrics["arm"].nunique()}')
metrics

In [ ]:
# Summary stats
summary = pd.read_csv('results/pilot/pilot_summary_stats.csv')
summary

In [ ]:
# Show comparison figure
from IPython.display import Image, display
import os

comp_path = 'figures/pilot/comparison_examples.png'
if os.path.isfile(comp_path):
    display(Image(filename=comp_path))
else:
    print(f'Comparison figure not found: {comp_path}')

In [ ]:
# Show Np vs Dice figure
np_dice_path = 'figures/pilot/np_vs_dice.png'
if os.path.isfile(np_dice_path):
    display(Image(filename=np_dice_path))
else:
    print(f'Np vs Dice figure not found: {np_dice_path}')

In [ ]:
# Show some debug overlays
import glob
from IPython.display import Image, display

debug_files = sorted(glob.glob('figures/pilot/debug_*.png'))[:6]
for f in debug_files:
    print(os.path.basename(f))
    display(Image(filename=f, width=400))

---
## 7. Package Outputs

In [ ]:
import shutil

output_zip = '/kaggle/working/phase2_outputs.zip'

# Zip results, figures, logs, notes
!cd /kaggle/working/AdaFoB && zip -r {output_zip} results/ figures/ logs/ notes/ \
    -x '*/masks/*'

print(f'\nOutput zip: {output_zip}')
print('Download from the Kaggle "Output" tab on the right sidebar.')
!ls -lh {output_zip}

In [ ]:
# Optional: Commit and push results back to GitHub
# Uncomment below if you have GITHUB_TOKEN set and want to push from Kaggle.

# import os
# from kaggle_secrets import UserSecretsClient
# secrets = UserSecretsClient()
# token = secrets.get_secret('GITHUB_TOKEN')
# 
# !cd /kaggle/working/AdaFoB && \
#   git config user.email 'pilot@adafob.local' && \
#   git config user.name 'AdaFoB Pilot Bot' && \
#   git add results/ figures/ logs/ notes/ && \
#   git commit -m 'Phase 2 pilot results from Kaggle' && \
#   git push https://{token}@github.com/PhoenixEvo/AdaFoB.git main
# print('Pushed results to GitHub.')